# Módulo 5 · Transformaciones con Glue Jobs (PySpark)


| # | Ejercicio | Equivalente en la CLI |
|---|---|---|
| 1 | Leer el script del job | — |
| 2 | Calcular localmente el resultado esperado (pandas) | — |
| 3 | Subir el script a `scripts/` | `aws s3 cp` |
| 4 | Crear el job `ventas-diarias` | `aws glue create-job` |
| 5 | Ejecutarlo para un día y monitorearlo | `aws glue start-job-run` + `get-job-runs` |
| 6 | Validar la salida en S3 frente al resultado esperado | `aws s3 ls` |
| 7 | (Opcional) Procesar más días | `start-job-run` con otra `--FECHA` |
| 8 | Registrar `ventas_diarias` en el catálogo y consultarla en Athena | `create-crawler` + Athena |
| 9 | Limpieza (opcional) | `delete-job`, `delete-crawler`, ... |

**Requisitos:** notebook 02 (rol `AWSGlueServiceRole-curso` y tabla `ventas_db.ventas`). Para el ejercicio 8 conviene el notebook 03 (workgroup `curso-de`).

**Costo:** Glue ETL cobra **USD 0.44 por DPU-hora**, por segundo y con **mínimo de 1 minuto**. Este job usa 2 workers G.1X (2 DPU) y tarda unos 2 minutos: **≈ USD 0.03 por ejecución**. Lo que más tarda es el arranque del cluster, no el procesamiento.

## Repaso: ¿qué hace un Glue Job?

```
 Tu script PySpark (en S3)  +  configuración del job (rol, versión, workers, argumentos)
            │
            ▼  start_job_run(--FECHA=2026-10-22)
 Glue levanta un cluster Spark sin servidores ── asume AWSGlueServiceRole-curso
            │
            ├─ lee   ventas_db.ventas  (solo la partición del día)   ← Data Catalog + S3 processed/
            ├─ agrega por canal y categoría                           ← PySpark
            └─ escribe Parquet en s3://<bucket>/curated/ventas_diarias/year=/month=/day=/
```

| Concepto | Valor en este job |
|---|---|
| **Glue version** | 5.0: Spark 3.5 y Python 3.11 |
| **Worker type** | G.1X = 1 DPU = 4 vCPU y 16 GB |
| **Número de workers** | 2 (el mínimo) |
| **DynamicFrame** | Formato propio de Glue. `toDF()` lo convierte en un DataFrame de Spark |

## Configuración

In [1]:
from pathlib import Path
import csv
import io
import json
import time

import boto3
import pandas as pd
from botocore.exceptions import ClientError, ProfileNotFound

BUCKET = "de-tecsup-marcell"            # <- el mismo del notebook 01
REGION = "us-east-2"
PERFIL = "curso"

DATABASE = "ventas_db"
ROL_GLUE = "AWSGlueServiceRole-curso"   # creado en el notebook 02
JOB = "ventas-diarias"
FECHA = "2026-10-22"                    # día a procesar

PROYECTO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
CREDENCIALES = PROYECTO / "credentials"
SCRIPT_LOCAL = PROYECTO / "glue_jobs" / "ventas_diarias.py"
SCRIPT_KEY = "scripts/ventas_diarias.py"
SALIDA_PREFIJO = "curated/ventas_diarias/"
STAGING = PROYECTO / "data" / "staging"

PRECIO_DPU_HORA = 0.44

In [8]:
def leer_csv_credenciales(carpeta: Path) -> dict:
    """Lee el .csv que descarga IAM (columnas 'Access key ID' y 'Secret access key')."""
    archivo = next(carpeta.glob("*accessKeys*.csv"))
    with archivo.open(encoding="utf-8-sig") as f:
        fila = next(csv.DictReader(f))
    return {
        "aws_access_key_id": fila["Access key ID"],
        "aws_secret_access_key": fila["Secret access key"],
    }


try:
    session = boto3.Session(profile_name=PERFIL, region_name=REGION)
    origen = f"perfil '{PERFIL}' de ~/.aws/"
except ProfileNotFound:
    session = boto3.Session(**leer_csv_credenciales(CREDENCIALES), region_name=REGION)
    origen = "archivo .csv de credentials/"

print("Credenciales tomadas de:", origen)
print("ARN:", session.client("sts").get_caller_identity()["Arn"])

s3 = session.client("s3")
iam = session.client("iam")
glue = session.client("glue")
athena = session.client("athena")
logs = session.client("logs")

# Requisitos del notebook 02
ROL_ARN = iam.get_role(RoleName=ROL_GLUE)["Role"]["Arn"]
glue.get_table(DatabaseName=DATABASE, Name="ventas")
print("Rol:", ROL_ARN)
print("Tabla ventas_db.ventas: OK")

Credenciales tomadas de: archivo .csv de credentials/
ARN: arn:aws:iam::073813807844:user/marcell-de
Rol: arn:aws:iam::073813807844:role/AWSGlueServiceRole-curso
Tabla ventas_db.ventas: OK


## Ejercicio 1 · El script del job

El script vive en `fundamentos-aws/glue_jobs/ventas_diarias.py`, versionado en Git como cualquier código. Tiene el esqueleto de la diapositiva 34:

| Paso | Qué hace |
|---|---|
| (1) `getResolvedOptions` | Lee los argumentos `--JOB_NAME`, `--BUCKET` y `--FECHA` |
| (2) `GlueContext` | Envuelve Spark y da acceso al Data Catalog |
| (3) `from_catalog(..., push_down_predicate=...)` | Lee **solo** la partición del día: filtra antes de tocar S3 |
| — | Agrega con PySpark puro: `groupBy` + `agg` |
| (4) `write.partitionBy(...).parquet(...)` | Escribe en `curated/` particionado por fecha |
| (5) `job.commit()` | Cierra el job |

`partitionOverwriteMode = dynamic` hace que `mode("overwrite")` reemplace **solo** la partición del día que se escribe, no toda la carpeta `curated/ventas_diarias/`.

> Este archivo **no** se ejecuta con el Python local: las librerías `awsglue` solo existen dentro de Glue.

In [3]:
print(SCRIPT_LOCAL.read_text())

"""Glue Job ventas-diarias.

Lee UNA partición (un día) de ventas_db.ventas desde el Data Catalog, la agrega
por canal y categoría, y escribe el resultado en la zona curated como Parquet
particionado por year/month/day.

Argumentos del job:
  --BUCKET  bucket del data lake (fijo: va en DefaultArguments del job)
  --FECHA   día a procesar, AAAA-MM-DD (cambia en cada ejecución)

Este archivo se ejecuta dentro de AWS Glue (Glue 5.0: Spark 3.5, Python 3.11).
Las librerías awsglue solo existen ahí: no lo ejecutes con el Python local.
"""

import sys

from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql import functions as F

# (1) Argumentos: Glue los pasa como --JOB_NAME, --BUCKET y --FECHA
args = getResolvedOptions(sys.argv, ["JOB_NAME", "BUCKET", "FECHA"])

# (2) Contexto: GlueContext envuelve Spark y da acceso al Data Catalog
glue = GlueContext(SparkContext.getOrCreate()

## Ejercicio 2 · Resultado esperado, calculado en local


Calculamos con pandas lo mismo que hará el job sobre el CSV de `staging/` del mismo día. Sale gratis y en el ejercicio 6 nos sirve para comprobar que Glue dio el mismo resultado.

In [9]:
ventas_dia = pd.read_csv(STAGING / FECHA / "ventas.csv")

esperado = (
    ventas_dia.groupby(["canal", "categoria"], as_index=False)
    .agg(pedidos=("pedido_id", "nunique"),
         unidades=("cantidad", "sum"),
         monto_total=("monto", "sum"))
)
esperado["monto_total"] = esperado["monto_total"].round(2)
esperado["ticket_promedio"] = (esperado["monto_total"] / esperado["pedidos"]).round(2)
esperado = esperado.sort_values(["canal", "categoria"], ignore_index=True)
esperado

,canal,categoria,pedidos,unidades,monto_total,ticket_promedio
0,tienda,Accesorios,10,29,5610.0,561.00
1,tienda,Libreria,9,28,180.0,20.00
2,tienda,Muebles,9,24,15360.0,1706.67
3,tienda,Tecnologia,18,50,67145.0,3730.28
4,web,Accesorios,13,37,7160.0,550.77
5,web,Libreria,13,29,212.5,16.35
6,web,Muebles,3,10,6000.0,2000.00
7,web,Tecnologia,15,38,46405.0,3093.67


## Ejercicio 3 · Subir el script a S3

Equivale a `aws s3 cp ventas_diarias.py s3://$BUCKET/scripts/`. Glue **siempre** lee el script desde S3, nunca desde tu máquina. Cada vez que cambies el `.py`, vuelve a ejecutar esta celda.

In [11]:
s3.upload_file(str(SCRIPT_LOCAL), BUCKET, SCRIPT_KEY)
SCRIPT_S3 = f"s3://{BUCKET}/{SCRIPT_KEY}"
print("subido:", SCRIPT_S3)

subido: s3://de-tecsup-marcell/scripts/ventas_diarias.py


## Ejercicio 4 · Crear el job

Equivale al `aws glue create-job` de la diapositiva 36.

| Parámetro | Valor | Para qué |
|---|---|---|
| `Role` | `AWSGlueServiceRole-curso` | Identidad con la que el job lee el catálogo y escribe en S3 |
| `Command` | `glueetl` + `ScriptLocation` | Job de Spark y dónde está su script |
| `GlueVersion` | `5.0` | Spark 3.5 y Python 3.11 |
| `WorkerType` / `NumberOfWorkers` | `G.1X` × 2 | 2 DPU, el mínimo |
| `DefaultArguments` | `--BUCKET` | Parámetros **fijos** del job. Los que cambian (`--FECHA`) se pasan en cada ejecución |
| `Timeout` | 10 minutos | Red de seguridad: si algo se cuelga, no sigue cobrando |
| `MaxRetries` | 0 | Si falla, no reintenta solo (y no cobra de nuevo) |

Los nombres de los argumentos llevan **doble guion**: `--BUCKET`, `--FECHA`.

In [12]:
definicion = dict(
    Role=ROL_GLUE,
    Command={"Name": "glueetl", "ScriptLocation": SCRIPT_S3, "PythonVersion": "3"},
    GlueVersion="5.0",
    WorkerType="G.1X",
    NumberOfWorkers=2,
    DefaultArguments={
        "--BUCKET": BUCKET,
        "--job-language": "python",
        "--enable-metrics": "true",
        "--enable-continuous-cloudwatch-log": "true",
    },
    Timeout=10,
    MaxRetries=0,
    ExecutionProperty={"MaxConcurrentRuns": 1},
    Description="Agrega una partición diaria de ventas por canal y categoría hacia curated/",
)

try:
    glue.create_job(Name=JOB, **definicion)
    print("Job creado:", JOB)
except glue.exceptions.AlreadyExistsException:
    glue.update_job(JobName=JOB, JobUpdate=definicion)
    print("El job ya existía: definición actualizada")

job = glue.get_job(JobName=JOB)["Job"]
print(f"{job['Name']} · Glue {job['GlueVersion']} · {job['NumberOfWorkers']} x {job['WorkerType']} · script: {job['Command']['ScriptLocation']}")

Job creado: ventas-diarias
ventas-diarias · Glue 5.0 · 2 x G.1X · script: s3://de-tecsup-marcell/scripts/ventas_diarias.py


## Ejercicio 5 · Ejecutar el job y monitorearlo

Equivale a `aws glue start-job-run --arguments '{"--FECHA": "2026-10-22"}'`.

Estados de un *job run*: `STARTING → RUNNING → SUCCEEDED`, o `FAILED` / `TIMEOUT` / `STOPPED`. **El arranque del cluster suele tardar más que el procesamiento.**

In [13]:
def ejecutar_job(fecha: str) -> dict:
    """Lanza el job para una fecha, espera a que termine y devuelve el job run."""
    run_id = glue.start_job_run(JobName=JOB, Arguments={"--FECHA": fecha})["JobRunId"]
    print(f"Job run {run_id} para --FECHA={fecha}")

    inicio, estado_anterior = time.time(), None
    while True:
        run = glue.get_job_run(JobName=JOB, RunId=run_id)["JobRun"]
        estado = run["JobRunState"]
        if estado != estado_anterior:
            print(f"[{time.time() - inicio:5.0f} s] {estado}")
            estado_anterior = estado
        if estado in ("SUCCEEDED", "FAILED", "TIMEOUT", "STOPPED", "ERROR"):
            return run
        time.sleep(15)


def resumen_run(run: dict) -> None:
    segundos = run.get("ExecutionTime", 0)
    dpu = run.get("MaxCapacity") or run["NumberOfWorkers"]          # G.1X: 1 DPU por worker
    costo = max(segundos, 60) / 3600 * dpu * PRECIO_DPU_HORA          # mínimo 1 minuto
    print(f"Estado:          {run['JobRunState']}")
    print(f"Tiempo cobrado:  {segundos} s × {dpu:g} DPU")
    print(f"Costo estimado:  USD {costo:.4f}")
    if run.get("ErrorMessage"):
        print("Error:", run["ErrorMessage"])


run = ejecutar_job(FECHA)
resumen_run(run)

Job run jr_be60cffda956490430dbe68358c8553600d0771343638bcc141112fb8e6b2e21 para --FECHA=2026-10-22
[    0 s] RUNNING
[   91 s] SUCCEEDED
Estado:          SUCCEEDED
Tiempo cobrado:  81 s × 2 DPU
Costo estimado:  USD 0.0198


### Si el job falla

El mensaje corto aparece arriba en `Error`. El detalle completo está en **CloudWatch Logs**:

| Grupo de logs | Contenido |
|---|---|
| `/aws-glue/jobs/output` | Lo que imprime el script: `print`, logs de Spark |
| `/aws-glue/jobs/error` | Errores y trazas de Python o Spark |

El *log stream* se llama igual que el `JobRunId`.

**Errores típicos:**
- `EntityNotFoundException`: la base o la tabla están mal escritas.
- `AccessDenied` en S3: al rol le falta la política inline del notebook 02.
- `Script not found`: no subiste el script (ejercicio 3).

> Tu usuario IAM no tiene permisos de CloudWatch Logs: no estaban en la lista de la diapositiva 10. Si quieres leer los logs desde aquí, ejecuta una vez la celda siguiente con `DAR_PERMISO_LOGS = True`. Adjunta la política administrada **de solo lectura** `CloudWatchLogsReadOnlyAccess` a tu usuario.

In [15]:
DAR_PERMISO_LOGS = True

if DAR_PERMISO_LOGS:
    usuario = session.client("sts").get_caller_identity()["Arn"].split("/")[-1]
    iam.attach_user_policy(
        UserName=usuario,
        PolicyArn="arn:aws:iam::aws:policy/CloudWatchLogsReadOnlyAccess",
    )
    print(f"CloudWatchLogsReadOnlyAccess adjuntada a {usuario}. Espera unos segundos antes de leer logs.")

CloudWatchLogsReadOnlyAccess adjuntada a marcell-de. Espera unos segundos antes de leer logs.


In [16]:
def mostrar_logs(run_id: str, grupo: str = "/aws-glue/jobs/output", ultimas: int = 30) -> None:
    try:
        eventos = logs.get_log_events(logGroupName=grupo, logStreamName=run_id,
                                      limit=ultimas, startFromHead=False)["events"]
    except ClientError as e:
        print(f"No se pudieron leer los logs ({e.response['Error']['Code']}): {e.response['Error']['Message']}")
        return
    for ev in eventos:
        print(ev["message"].rstrip())


# Busca la línea "Filas agregadas para ..." que imprime el script
mostrar_logs(run["Id"])
# Si el job falló: mostrar_logs(run["Id"], "/aws-glue/jobs/error")

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
Filas agregadas para 2026-10-22: 8
Running autoDebugger shutdown hook.


## Ejercicio 6 · Validar la salida

Primero vemos qué escribió el job en S3. Spark escribe uno o varios archivos `part-*.parquet` por partición.

In [17]:
y, m, d = FECHA.split("-")
prefijo_dia = f"{SALIDA_PREFIJO}year={y}/month={m}/day={d}/"

archivos = [o["Key"] for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=SALIDA_PREFIJO).get("Contents", [])]
for k in archivos:
    print(k)

curated/ventas_diarias/year=2026/month=10/day=22/part-00000-7367d28d-adb8-4fd9-97a6-3a25611b5d07.c000.snappy.parquet


Ahora leemos la partición del día y la comparamos con el resultado esperado del ejercicio 2. Si Glue y pandas calcularon lo mismo, la celda termina con ✅.

In [18]:
partes = [
    pd.read_parquet(io.BytesIO(s3.get_object(Bucket=BUCKET, Key=k)["Body"].read()))
    for k in archivos if k.startswith(prefijo_dia) and k.endswith(".parquet")
]
obtenido = (
    pd.concat(partes, ignore_index=True)
    [["canal", "categoria", "pedidos", "unidades", "monto_total", "ticket_promedio"]]
    .sort_values(["canal", "categoria"], ignore_index=True)
)

comparacion = esperado.merge(obtenido, on=["canal", "categoria"], suffixes=("_local", "_glue"), how="outer")
for col in ["pedidos", "unidades", "monto_total", "ticket_promedio"]:
    # tolerancia de 0.01 por posibles diferencias de redondeo entre Spark y pandas
    assert ((comparacion[f"{col}_local"] - comparacion[f"{col}_glue"]).abs() <= 0.01).all(), f"Diferencia en {col}"

print(f"✅ Glue y pandas coinciden en las {len(obtenido)} filas de {FECHA}")
obtenido

✅ Glue y pandas coinciden en las 8 filas de 2026-10-22


,canal,categoria,pedidos,unidades,monto_total,ticket_promedio
0,tienda,Accesorios,10,29,5610.0,561.00
1,tienda,Libreria,9,28,180.0,20.00
2,tienda,Muebles,9,24,15360.0,1706.67
3,tienda,Tecnologia,18,50,67145.0,3730.28
4,web,Accesorios,13,37,7160.0,550.77
5,web,Libreria,13,29,212.5,16.35
6,web,Muebles,3,10,6000.0,2000.00
7,web,Tecnologia,15,38,46405.0,3093.67


## Ejercicio 7 · (Opcional) Procesar más días

Cada ejecución procesa **un** día y cuesta ≈ USD 0.03. Agrega fechas a la lista para llenar `curated/` con más días. Las ejecuciones van una tras otra porque el job admite 1 ejecución a la vez (`MaxConcurrentRuns = 1`).

Gracias a `partitionOverwriteMode = dynamic`, cada ejecución reemplaza **solo su día**. Si vuelves a procesar `2026-10-22`, el resto de días queda intacto. Así funcionará cuando Airflow lance el job cada día (módulo 6).

In [ ]:
FECHAS_EXTRA = ["2026-10-22"]   # ej. ["2026-10-20", "2026-10-21", "2026-10-23"]

for fecha in FECHAS_EXTRA:
    r = ejecutar_job(fecha)
    resumen_run(r)
    print()
    time.sleep(5)

print(sorted({k.split("/")[-2] for k in
              (o["Key"] for o in s3.list_objects_v2(Bucket=BUCKET, Prefix=SALIDA_PREFIJO).get("Contents", []))
              if "day=" in k}))

## Ejercicio 8 · Registrar `ventas_diarias` y consultarla en Athena

El job escribió archivos, pero el catálogo todavía no sabe que existen. Igual que en el notebook 02, un crawler registra la tabla

In [23]:
CRAWLER_DIARIAS = "crawler-ventas-diarias"

config_crawler = dict(
    Name=CRAWLER_DIARIAS,
    Role=ROL_GLUE,
    DatabaseName=DATABASE,
    Targets={"S3Targets": [{
        "Path": f"s3://{BUCKET}/{SALIDA_PREFIJO}",
        "Exclusions": ["**/_SUCCESS", "**_$folder$"],   # archivos de control de Spark, no son datos
    }]},
    SchemaChangePolicy={"UpdateBehavior": "UPDATE_IN_DATABASE", "DeleteBehavior": "LOG"},
)
try:
    glue.create_crawler(**config_crawler)
    print("Crawler creado:", CRAWLER_DIARIAS)
except glue.exceptions.AlreadyExistsException:
    glue.update_crawler(**config_crawler)
    print("El crawler ya existía: configuración actualizada")

glue.start_crawler(Name=CRAWLER_DIARIAS)
inicio, estado_anterior = time.time(), None
while True:
    estado = glue.get_crawler(Name=CRAWLER_DIARIAS)["Crawler"]["State"]
    if estado != estado_anterior:
        print(f"[{time.time() - inicio:5.0f} s] {estado}")
        estado_anterior = estado
    if estado == "READY" and time.time() - inicio > 5:
        break
    time.sleep(10)

print("Resultado:", glue.get_crawler(Name=CRAWLER_DIARIAS)["Crawler"]["LastCrawl"]["Status"])
print("Tablas en", DATABASE, "->", [t["Name"] for t in glue.get_tables(DatabaseName=DATABASE)["TableList"]])

Crawler creado: crawler-ventas-diarias
[    0 s] RUNNING
[   41 s] READY
Resultado: SUCCEEDED
Tablas en ventas_db -> ['resumen_cliente', 'v_web', 'ventas', 'ventas_diarias']


Consultamos la nueva tabla con Athena. Si hiciste el notebook 03 se usa el workgroup `curso-de`; si no, el `primary`, indicándole dónde guardar los resultados.

In [24]:
def consultar(sql: str) -> pd.DataFrame:
    workgroups = [w["Name"] for w in athena.list_work_groups()["WorkGroups"]]
    params = dict(QueryString=sql, QueryExecutionContext={"Database": DATABASE})
    if "curso-de" in workgroups:
        params["WorkGroup"] = "curso-de"
    else:
        params["ResultConfiguration"] = {"OutputLocation": f"s3://{BUCKET}/athena-results/"}

    qid = athena.start_query_execution(**params)["QueryExecutionId"]
    while (estado := athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"])["State"] in ("QUEUED", "RUNNING"):
        time.sleep(1)
    if estado["State"] != "SUCCEEDED":
        raise RuntimeError(estado.get("StateChangeReason"))

    filas = athena.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
    valores = [[c.get("VarCharValue") for c in f["Data"]] for f in filas]
    return pd.DataFrame(valores[1:], columns=valores[0])


consultar(f"""
SELECT canal, categoria, pedidos, monto_total, ticket_promedio
FROM ventas_db.ventas_diarias
WHERE year = '{y}' AND month = '{m}' AND day = '{d}'
ORDER BY monto_total DESC
""")

,canal,categoria,pedidos,monto_total,ticket_promedio
0,tienda,Tecnologia,18,67145.0,3730.28
1,web,Tecnologia,15,46405.0,3093.67
2,tienda,Muebles,9,15360.0,1706.67
3,web,Accesorios,13,7160.0,550.77
4,web,Muebles,3,6000.0,2000.0
5,tienda,Accesorios,10,5610.0,561.0
6,web,Libreria,13,212.5,16.35
7,tienda,Libreria,9,180.0,20.0


Así queda el data lake después de los módulos 2 a 5:

| Zona | Prefijo en S3 | Tabla en el catálogo | La crea |
|---|---|---|---|
| raw | `raw/clientes/` | — | notebook 01 |
| processed | `processed/ventas/` | `ventas` | notebook 01 + crawler (02) |
| curated | `curated/resumen_cliente/` | `resumen_cliente` | CTAS de Athena (03) |
| curated | `curated/ventas_diarias/` | `ventas_diarias` | **Glue Job** + crawler (04) |

## Ejercicio 9 · Limpieza (opcional)

**No lo ejecutes todavía:** en el módulo 6, Airflow lanzará este mismo job. Al terminar el curso, cambia `BORRAR` a `True`.

Borra el job, el crawler, la tabla `ventas_diarias` del catálogo, sus archivos en `curated/` y el script. No toca el rol ni la tabla `ventas` (notebook 02).

> **Orden de limpieza:** de atrás hacia adelante, **04 → 03 → 02 → 01**. El bucket (notebook 01) se borra al final.

In [7]:
BORRAR = True
CRAWLER_DIARIAS = "crawler-ventas-diarias"


def borrar_prefijo(prefijo: str) -> None:
    try:
        for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefijo):
            objetos = [{"Key": o["Key"]} for o in pagina.get("Contents", [])]
            if objetos:
                s3.delete_objects(Bucket=BUCKET, Delete={"Objects": objetos})
        print("Archivos borrados en", prefijo)
    except ClientError as e:
        if e.response["Error"]["Code"] != "NoSuchBucket":
            raise
        print(f"El bucket {BUCKET} ya no existe: nada que borrar en {prefijo}")


if BORRAR:
    # delete_job no falla si el job no existe
    glue.delete_job(JobName=JOB)
    print("Job eliminado:", JOB)

    try:
        glue.delete_crawler(Name=CRAWLER_DIARIAS)
        print("Crawler eliminado:", CRAWLER_DIARIAS)
    except glue.exceptions.EntityNotFoundException:
        print("El crawler ya no existía:", CRAWLER_DIARIAS)

    try:
        glue.delete_table(DatabaseName=DATABASE, Name="ventas_diarias")
        print("Borrada del catálogo: ventas_diarias")
    except glue.exceptions.EntityNotFoundException:
        print("Ya no existía en el catálogo: ventas_diarias")

    borrar_prefijo(SALIDA_PREFIJO)
    borrar_prefijo("scripts/")

Job eliminado: ventas-diarias
El crawler ya no existía: crawler-ventas-diarias
Borrada del catálogo: ventas_diarias
Archivos borrados en curated/ventas_diarias/
Archivos borrados en scripts/
